# Vaudeville Structured Data with LLM

- Notebook by Daniel Russo-Batterham, Charlie Cross, Richard Freedman, Miles Fagan.
- Last updated November 17, 2025

## 0.  Set up File Paths and Imports

In [1]:
# initial imports of libraries


from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
# from langchain.document_loaders import PDFPlumberLoader
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

from typing import Optional, List
from typing_extensions import List, TypedDict, Optional
from langchain_core.prompts import ChatPromptTemplate

import getpass
import os
import csv

from pydantic import BaseModel, Field

async def loadPDF(filepath: str) -> list:
    loader = PyPDFLoader(filepath)
    pages = []
    async for page in loader.alazy_load():
        pages.append(page)   
    return pages

In [2]:
# path to your pdf file
pdf_filepath = "PDFs/Coraly_ou_la_soeur_et_le_Frere.pdf" 


In [3]:
# location of your output csv file
csv_filename = pdf_filepath.replace("PDFs/","").replace(".pdf",".csv") 

In [4]:

# load PDF
source = await loadPDF(pdf_filepath)

for page in source:
        page.metadata['source'] = page.metadata['source'].replace("Files\PDFs\\","")

source_content = ""
for page in source:
    source_content += page.page_content
source_full = Document(page_content = source_content, metadata = source[0].metadata)

### Select LLM and Provide API Key

- You can choose your preferred LLM here. Make sure to have the necessary API keys set up in your environment.



In [5]:
# set up LLM model. Prompt for OpenAI key if not set in environment.  Just click "return" after you enter the key in the box

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5", model_provider="openai")
processing_llm = init_chat_model("gpt-5", model_provider="openai")

Enter API key for OpenAI:  ········


## 1.  Setting up Classes for Pydantic

In [6]:
# Our first task involves dividing the pdf into scenes.  We do this by prompting the LLM to identify scene headers and labels

class Scene(BaseModel):
    """A single scene from a Vaudeville play"""

    act: int = Field(description="The act number or label as it appears in the text.")
    scene: int = Field(description="The scene number or label as it appears in the text.")
    header: str = Field(description="The exact scene header line, copied verbatim from the text.")

class FullPlay(BaseModel):
    """A full play, that has yet to be broken into individual scenes."""

    all_scenes: List[Scene] = Field(description="A list of every single scene's header and label - each as a Scene object.")

formatted_splitter_llm = processing_llm.with_structured_output(FullPlay)

In [7]:
# system prompt for scene splitting--it seems to work well, but may need to be adjusted for different plays or printers

prompt = f"""
The following is the full text of a French Vaudeville play. Your job is to identify every scene boundary.
For each scene, return:
- The act number (as it appears in the text)
- The scene number (as it appears in the text)
- The exact scene header line (copy it verbatim from the text)

Return a list of objects like:
{{"act": "...", "scene": "...", "header": "..."}}

Do not attempt to count character indexes. Only return the scene headers as they appear in the text.

Play Content: \n
{source_full.page_content}
"""

In [8]:
# function to split up play into scenes
def split_up_play(doc):
    response = formatted_splitter_llm.invoke(prompt)
    return response

In [9]:
# run the functino on the full play we loaded earlier
all_indexes = split_up_play(source_full)

In [10]:
# pass the split scenes and put them in a list called 'all_splits'

all_splits = []
scene_headers = all_indexes.all_scenes
full_text = source_full.page_content

prev_end_idx = 0
for i, scene in enumerate(scene_headers):
    # Find the start index of this scene's header after the previous end index
    start_idx = full_text.find(scene.header, prev_end_idx)
    if start_idx == -1:
        print(f"Scene header not found: {scene.header}")

    # Determine the end index: start of next scene header, or end of document
    if i + 1 < len(scene_headers):
        next_start_idx = full_text.find(scene_headers[i + 1].header, start_idx + len(scene.header))
        if next_start_idx == -1:
            end_idx = len(full_text)
        else:
            end_idx = next_start_idx
    else:
        end_idx = len(full_text)

    scene_text = full_text[start_idx:end_idx]
    doc = Document(page_content=scene_text, metadata={"act": scene.act, "scene": scene.scene, "header": scene.header})
    all_splits.append(doc)
    prev_end_idx = end_idx

## 2.  The Musical Moment Class

Once we have the scenes, we can start to analyze them for musical moments.  We set up a class for musical moments.
This Pydantic class captures the key elements of a musical moment in a Vaudeville play, and returns as structured data.

- act
- scene
- number
- characters
- dramatic situation
- air or melodie
- poetic text
- rhyme scheme
- stage directions

For the moment we do NOT include versification, but that could be added later, perhaps with the help of a specialized Python versification library.

In [11]:


# pydantic class for Musical Moment
class MusicalMoment(BaseModel):
    """Many of these musical moments reuse some preexisting (and often well-known)  melody or tune.  These are variously called "melodie”, or “air”, and identified with a short title that refers in some way to an opera or collection of melodies from which it was drawn.  The titles might include the names of works, or other characters in those original works. In the context of the plays, these tunes become the vehicle for newly composed lyrics, which are normally rhymed, and which normally follow the poetic scansion and structure of the original lyrics.  Rhyme, versification and structure are thus of interest to us."""

    act: int = Field(description="The act number in which this musical moment takes place. Will be labeled at the top of the act or scene in which it takes place.")
    scene: int = Field(description="The scene number in which the musical moment takes place. Will be labeled at the top of the scene.")
    number: int = Field(description = "The index of the musical moment in the scene. For example, if this is the first musical moment in the scene, this should be 1.")
    characters: list[str] = Field(description="the character or characters who are singing (or otherwise making music) within this specific musical moment,")
    dramatic_situation: str = Field(description="the dramatic situation (a love scene, a crowd scene) in which the musical moment is occurring")

    air_or_melodie: str = Field(description="The title of the 'air' or 'melodie' of which the musical moment is based. It will be labeled in the text as 'air' or 'melodie'.")
    # now with Freedman's edits to seek character-specific lines
    poetic_text: str = Field(description="The text from the music number. Do not include stage directions, only the lyrics sung by the characters in this musical moment.  Note that sometimes the text of a given number will be sung by different characters.  Tell us which character sings which lines, but do not include the names of the characters in the transcribed poetry itself.  ")
    rhyme_scheme: str = Field(description = "The rhyme scheme for the poetic text in the musical moment. For example, sentences that end in 'tree' 'be' 'why' and 'high' would have a rhyme scheme of AABB.  If there are multiple parts with different rhyme schemes, please indicate which part has which scheme.  If more than one character is singing, please indicate which character's lines correspond to which rhyme scheme.")
    stage_direction_or_cues: Optional[str] = Field(description="any stage directions, which tell a character what to do, but aren't a part of another character's dialogue. These are usually connected with a character’s name, and often are in some contrasting typography (italics, or in parentheses - though this may not be picked up by the filereader).  Sometimes these directions even happen in the midst of a song! In a related way there are sometimes ‘cues’ for music, or performance (as when there is an offstage sound effect, or someone is humming) Most times the stage directions appear just before or after the song text. But sometimes they appear in the midst of the texts. The directions should be reported here and not in the transcription of the poem.")

class VaudevillePlay(BaseModel):
    musicalMoments: list[MusicalMoment] = Field(description="""A list of musical moments in a Vaudeville play, as MusicalMoment objects. Many of these musical moments reuse some preexisting (and often well-known)  melody or tune.  These are variously called "melodie”, or “air”, and identified with a short title that refers in some way to an opera or collection of melodies from which it was drawn.  The titles might include the names of works, or other characters in those original works. In the context of the plays, these tunes become the vehicle for newly composed lyrics, which are normally rhymed, and which normally follow the poetic scansion and structure of the original lyrics.  Rhyme, versification and structure are thus of interest to us.""")    

structured_llm = llm.with_structured_output(VaudevillePlay)

## The System Prompt

In [12]:
# and the general system and human prompts for extracting musical moments--it seems to work well, but may need to be adjusted for different plays or printers
# note that some of the prompt text might conflict with the pydantic class definitions above!  

system_prompt = """
You are a literary analyst specializing in French Vaudeville plays from the 19th century. 
Your goal is to identify each musical moment in the text, and for each, extract detailed structured information, 
including act, scene, characters, air or melodie, poetic text, and rhyme scheme. 
Some parts of the text were slightly misinterpreted by the file reader (e.g., missing spaces or strange line breaks).
Many songs look to have breaks in the middle when different characters have different parts - don't split up one song into multiple parts just because multiple characeters have different parts.
Reminder that the scenes are labeled using Roman numerals (I is one, II is two, etc).
Check that the songs you've identified are actually songs in the play be ensuring that only the characters that appear in the play have parts in the song.
"""
human_prompt = """
Given the following chunk of the play, analyze and return the musical moments as a structured VaudevillePlay object.
"""

## 3. Build the Lang Graph to Extract Musical Moments

In [13]:
# don't edit these

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","Context:\n{context}\n\nQuestion:\n{question}")
     ])

class State(TypedDict):
    index: int
    context: Document
    answer: str

def check_index(state: State):
    return state

def retrieve_doc(state: State):
    document = all_splits[state["index"]]
    return {"context": document}

def generate(state: State):
    i = state["index"]
    message = prompt.invoke({"question":human_prompt,"context" : f'Act {all_indexes.all_scenes[i].act}, Scene {all_indexes.all_scenes[i].scene}:\n\n {state["context"].page_content}'})
    response = structured_llm.invoke(message)
    return {"answer": response}

from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([check_index, retrieve_doc, generate])
graph_builder.add_edge(START, "check_index")
graph = graph_builder.compile()

In [ ]:
# function to analyze all scenes and extract musical moments--we pass 'all_splits' into it
def analyze_scenes(docs: List[Document]) -> List[MusicalMoment]:
    all_moments: List[MusicalMoment] = []

    for i,doc in enumerate(docs):
        response = graph.invoke({"index": i})
        moments = response["answer"].musicalMoments
        all_moments.extend(moments)
    
    return all_moments

all_moments = analyze_scenes(all_splits[0:5])

## 4. Output

Note that we output both to CSV and JSON formats for ease of use in different applications.
The JSON will preserve nested structures better, while the CSV is easier to open in spreadsheet software.


- 'moments_dicts' will be a list of dictionaries, each representing a musical moment extracted from the play.


In [ ]:

# Convert all MusicalMoment objects to dicts for CSV output
moments_dicts = [moment.model_dump() for moment in all_moments]

# Get all field names from the first moment
fieldnames = moments_dicts[0].keys() if moments_dicts else []

# Write to CSV
# Ensure the output folder exists
output_folder = "csv_outputs"
os.makedirs(output_folder, exist_ok=True)

# Build the output path
output_path = os.path.join(output_folder, os.path.basename(csv_filename))

with open(output_path, "w", newline='', encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in moments_dicts:
        # Convert lists to strings for CSV output
        for key, value in row.items():
            if isinstance(value, list):
                row[key] = "; ".join(str(v) for v in value)
        writer.writerow(row)